In [20]:
import os
from datasets import load_dataset

FILE_CORPUS = "corpus_wikipedia.txt"
N_ARTICLES = 300

# Scarica da Hugging Face SOLO se il file locale non esiste già
if not os.path.exists(FILE_CORPUS):
    print("Download del dataset da Hugging Face in corso...")
    dataset = load_dataset(
        "wikimedia/wikipedia",
        "20231101.it",
        split=f"train[:{N_ARTICLES}]"
    )
    corpus_wiki = "\n".join(dataset["text"])

    # Salva il corpus su un file locale .txt
    with open(FILE_CORPUS, "w", encoding="utf-8") as f:
        f.write(corpus_wiki)
    print("Corpus salvato localmente!")
else:
    print("Caricamento del corpus dal file locale...")
    # Legge direttamente dal file locale in memoria (istantaneo e senza rete)
    with open(FILE_CORPUS, "r", encoding="utf-8") as f:
        corpus_wiki = f.read()

parole_totali = len(corpus_wiki.split())
print(f"Parole totali (grezze): {parole_totali}")

Caricamento del corpus dal file locale...
Parole totali (grezze): 348820


In [27]:
word2idx, idx2word, word_freq = build_vocab(corpus_wiki)
corpus_to_idx = encode(corpus_wiki, word2idx)
pairs = generate_training_pairs(corpus_to_idx, 3,idx2word)
pairs_arr = np.array(pairs)
targets = pairs_arr[:, 0]
contexts = pairs_arr[:, 1]


# Parametri
batch_size = 1024
num_samples = len(targets)

vocab_size = len(word2idx)
embedding_dim = 50
epochs = 5
learning_rate = 0.3

# Inizializzo la rete
embed_layer = EmbeddingLayer(vocab_size, embedding_dim)
dense_layer = DenseLayer(embedding_dim, vocab_size)
loss_layer = SoftMaxCrossEntropy()

# Creazione sets
perm = np.random.permutation(num_samples)
shuffled_targets = targets[perm]
shuffled_contexts = contexts[perm]

idx_split = int(np.ceil(num_samples * 0.9))
training_targets = shuffled_targets[:idx_split]
testing_targets = shuffled_targets[idx_split:]
training_contexts = shuffled_contexts[:idx_split]
testing_contexts = shuffled_contexts[idx_split:]

# Training

def to_one_hot(indices, vocab_size):
    oh = np.zeros((len(indices), vocab_size))
    oh[np.arange(len(indices)), indices] = 1.0
    return oh

num_samples_training = len(training_targets)  # ora è il 90%, non il 100%
num_batches_per_epoch = int(np.ceil(num_samples_training / batch_size))

for epoch in range(epochs):
  perm = np.random.permutation(idx_split)
  shuffled_targets = training_targets[perm]
  shuffled_contexts = training_contexts[perm]

  epoch_loss = 0.0

  for b_idx, start in enumerate(range(0, num_samples_training, batch_size)):
    end = min(start + batch_size, num_samples_training)

    batch_targets = shuffled_targets[start:end]
    batch_contexts = shuffled_contexts[start:end]
    y_true = to_one_hot(batch_contexts, vocab_size)

    # Forward pass
    h = embed_layer.forward(batch_targets)
    z = dense_layer.forward(h)
    loss = loss_layer.forward(z, y_true)
    # Backward pass
    grad_loss = loss_layer.backward()
    grad_dense = dense_layer.backward(grad_loss)
    embed_layer.backward(grad_dense)

    # Gradient update
    dense_layer.w -= learning_rate * dense_layer.dw
    dense_layer.b -= learning_rate * dense_layer.db
    embed_layer.w -= learning_rate * embed_layer.dw

    epoch_loss += loss

  avg_loss = epoch_loss / num_batches_per_epoch
  print(f"Epoch {epoch+1}/{epochs} - Loss media: {avg_loss:.4f} ")

W_in = embed_layer.w

Epoch 1/5 - Loss media: 4.0591 
Epoch 2/5 - Loss media: 3.8757 
Epoch 3/5 - Loss media: 3.8561 
Epoch 4/5 - Loss media: 3.8470 
Epoch 5/5 - Loss media: 3.8407 


In [29]:
def evaluate_topk_accuracy(test_targets, test_contexts, W_in, idx2word,
                            top_k=10, n_samples=200, n_examples_to_show=8):
    """
    Valuta il top-k accuracy sul test set e mostra alcuni esempi
    rappresentativi (hit e miss) per ispezione qualitativa.
    """
    norms = np.linalg.norm(W_in, axis=1, keepdims=True) + 1e-9
    W_norm = W_in / norms

    n_samples = min(n_samples, len(test_targets))
    sample_idx = np.random.choice(len(test_targets), n_samples, replace=False)

    hits = 0
    examples = []  # accumuliamo qui, non stampiamo dentro il ciclo

    for i in sample_idx:
        target_idx = test_targets[i]
        context_idx = test_contexts[i]

        vec = W_norm[target_idx]
        sims = W_norm @ vec
        top_indices = np.argsort(sims)[::-1]
        top_indices = top_indices[top_indices != target_idx][:top_k]

        is_hit = context_idx in top_indices
        if is_hit:
            hits += 1

        examples.append({
            "target": idx2word[target_idx],
            "context_reale": idx2word[context_idx],
            "hit": is_hit,
            "top_previsti": [idx2word[j] for j in top_indices[:5]],  # solo i primi 5 per leggibilità
        })

    accuracy = hits / n_samples

    # --- Riepilogo numerico ---
    print(f"Top-{top_k} accuracy sul test set: {accuracy:.2%} (su {n_samples} campioni)\n")

    # --- Esempi rappresentativi: metà hit, metà miss, per un confronto equilibrato ---
    hit_examples = [e for e in examples if e["hit"]][:n_examples_to_show // 2]
    miss_examples = [e for e in examples if not e["hit"]][:n_examples_to_show // 2]

    print(f"--- Esempi di HIT (contesto reale tra i top-{top_k}) ---")
    for e in hit_examples:
        print(f"  target: {e['target']:<15} contesto reale: {e['context_reale']:<15} "
              f"top-5 previsti: {e['top_previsti']}")

    print(f"\n--- Esempi di MISS (contesto reale fuori dai top-{top_k}) ---")
    for e in miss_examples:
        print(f"  target: {e['target']:<15} contesto reale: {e['context_reale']:<15} "
              f"top-5 previsti: {e['top_previsti']}")

    return accuracy

evaluate_topk_accuracy(testing_targets, testing_contexts, W_in, idx2word, top_k=10, n_samples=200)

Top-10 accuracy sul test set: 14.00% (su 200 campioni)

--- Esempi di HIT (contesto reale tra i top-10) ---
  target: non             contesto reale: l               top-5 previsti: ['si', 'in', 'l', 'da', 'anche']
  target: <UNK>           contesto reale: e               top-5 previsti: ['del', 'nel', 'e', 'in', 'sono']
  target: di              contesto reale: per             top-5 previsti: ['in', 'a', 'è', 'dei', 'da']
  target: nel             contesto reale: <UNK>           top-5 previsti: ['le', 'dei', '<UNK>', 'l', 'gli']

--- Esempi di MISS (contesto reale fuori dai top-10) ---
  target: ancona          contesto reale: <UNK>           top-5 previsti: ['l', 'principali', 'al', 'la', 'i']
  target: <UNK>           contesto reale: <UNK>           top-5 previsti: ['del', 'nel', 'e', 'in', 'sono']
  target: all             contesto reale: <UNK>           top-5 previsti: ['due', 'da', 'essere', 'a', 'fu']
  target: l               contesto reale: <UNK>           top-5 previsti: ['la

0.14

In [4]:
import numpy as np
import re

def tokenize_text(corpus):
  new_corpus = re.sub(r"[\n.!?]", " NWL ", corpus.lower())
  new_corpus = re.sub(r"[^\w\s]", " ", new_corpus)
  new_corpus = re.sub(r" NWL ", " <NWL> ", new_corpus)
  return new_corpus

def build_vocab(corpus):
  # Tokenizza il testo, conta le frequenze
  # Ritorna: word2idx, idx2word, word_freq

  new_corpus = tokenize_text(corpus)
  words = new_corpus.split()

  n = len(words)
  min_freq = int(n * 0.001) # tengolo solo le parole che compaiono almeno 1 volta ogni 1000 parole es. testo da 5000 parole, tengolo solo quelle che compaiono almeno 5 volte, con limite a 10
  if min_freq > 50:
    min_freq = 50

  unique_words, frequences = np.unique(words, return_counts=True)

  filtered_words = unique_words[frequences >= min_freq]
  filtered_freqs = frequences[frequences >= min_freq]

  word2idx = {}
  idx2word = {}
  word_freq = {}

  nwl_idx = np.where(filtered_words == "<NWL>")[0]
  unk_idx = np.where(frequences < min_freq)[0]
  nwl_frq = filtered_freqs[nwl_idx]
  unk_frq = np.sum(frequences[unk_idx])

  word_freq["<NWL>"] = nwl_frq[0]
  word2idx["<NWL>"] = 0
  idx2word[0] = "<NWL>"
  filtered_words = np.delete(filtered_words, nwl_idx, 0)
  filtered_freqs = np.delete(filtered_freqs, nwl_idx, 0)

  for idx, word in enumerate(filtered_words):
    word2idx[str(word)] = idx + 1
    idx2word[idx + 1] = str(word)
    word_freq[str(word)] = filtered_freqs[idx]

  last_idx = len(filtered_words) + 1
  word_freq["<UNK>"] = unk_frq
  word2idx["<UNK>"] = last_idx
  idx2word[last_idx] = "<UNK>"

  return word2idx, idx2word, word_freq

def encode(corpus, word2idx):
  out = []

  new_corpus = tokenize_text(corpus)
  words = new_corpus.split()

  for word in words:
    try:
      out.append(word2idx[word])
    except KeyError:
      out.append(word2idx["<UNK>"])
  return out

def decode(corpus_indexes, idx2word):
  out = []
  for idx in corpus_indexes:
      out.append(idx2word[idx])
  return out


def generate_training_pairs(corpus_indices, window_size=3, idx2word=None):
    # Per ogni parola nel corpus, genera coppie (target, context)
    # scorrendo una finestra di ±window_size

    # Output atteso: lista di coppie (target_idx, context_idx)
    # Attenzione ai bordi delle frasi/testo (non puoi prendere contesto "fuori" dal testo)

    out = []
    last_idx = len(corpus_indices) - 1

    for pos, idx in enumerate(corpus_indices):

      if idx != 0:
        for j in range(1, window_size + 1): # sx
            if (pos - j) >= 0:
              out.append([idx, corpus_indices[pos - j]])
              if corpus_indices[pos - j] == 0: # if context == '<NWL>' break the cicle for that word
                break

        for j in range(1, window_size + 1): # dx
          if (pos + j) <= last_idx:
            out.append([idx, corpus_indices[pos + j]])

            if corpus_indices[pos + j] == 0: # if context == '<NWL>' break the cicle for that word
              break

    return out


class Layer:
  def forward(self, inputs):
      raise NotImplementedError
  def backward(self, output_gradient):
      raise NotImplementedError

class EmbeddingLayer(Layer):
  def __init__(self, vocab_size, embedding_dim):
    self.w = np.random.randn(vocab_size, embedding_dim) * np.sqrt(2.0 / vocab_size) # w_in (vocab_size, embedding_dim)
    self.dw = np.zeros(self.w.shape)
    self.input = None

  def forward(self, inputs):
    self.input = inputs
    return self.w[inputs] # (input_len, embedding_dim)

  def backward(self, output_gradient):
    # output_gradient       (input_len, embedding_dim)
    # dw                    (vocab_size, embedding_dim)
    # input                 (input_len)
    # self.dw[self.input]   (input_len, embedding_dim)
    self.dw = np.zeros_like(self.w)
    for idx, word in enumerate(self.input):
      self.dw[word] += output_gradient[idx]

class DenseLayer(Layer):
  def __init__(self, fan_in, fan_out):
    self.w = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_out) # (embedding_dim, vocab_size)
    self.b = np.zeros((fan_out, 1))  # (vocab_size, 1)
    self.dw = None
    self.db = None
    self.input = None

  def forward(self, inputs):
    self.input = inputs # (input_len, embedding_dim)
    z = inputs @ self.w + self.b.T  # (input_len, embedding_dim) @ (embedding_dim, vocab_size) = (input_len, vocab_size)
    return z # (input_len, vocab_size)

  def backward(self, out_grad):
    # out_grad (input_len, vocab_size)
    N = self.input.shape[1] # embedding_dim

    self.db =  np.sum(out_grad, axis=0, keepdims=True).T # (vocab_size, 1)
    self.dw =  self.input.T @ out_grad        # (embedding_dim, vocab_size)

    return out_grad @ self.w.T  # (input_len, embedding_dim)

class SoftMaxCrossEntropy(Layer):
  def __init__(self):
    self.output = None
    self.y_true = None

  def forward(self, inputs, y_true):
    self.y_true = y_true  # (input_len, vocab_size)

    # 1. Softmax su axis=1 (lungo le classi)
    shifted_inputs = inputs - np.max(inputs, axis=1, keepdims=True)
    exps = np.exp(shifted_inputs)
    self.output = exps / np.sum(exps, axis=1, keepdims=True)

    # 2. Cross Entropy Loss corretta
    eps = 1e-15
    clipped_values = np.clip(self.output, eps, 1 - eps)

    # np.sum su axis=1 isola la probabilità della parola corretta per OGNI campione.
    # np.mean fa la media tra tutti i campioni del batch.
    loss_per_sample = -np.sum(self.y_true * np.log(clipped_values), axis=1)
    return np.mean(loss_per_sample)

  def backward(self, out_grad=None):
    # gradiente analitico dZ = (P - Y) / batch_size
    batch_size = self.y_true.shape[0]
    return (self.output - self.y_true) / batch_size

